# PCB Defect Detection – Kaggle Training Notebook

**Environment:** Kaggle GPU (2× T4 15 GB or P100 16 GB) · 32 GB RAM  
**Assumption:** The project repository is available at `/kaggle/working/project`.  
Specifically, the directory layout expected inside that folder is:
```
/kaggle/working/project/
  classes.txt
  scripts/
    train_kaggle.py
    requirements_train.txt
    yolo_data.yaml
    mmdet_configs/
  data/           <- or symlinked from Kaggle input dataset
    train/images/
    val/images/
    test/images/
    annotations_json/
```

Edit the **Configuration** cell below to control which models are trained.

## 1 · Environment check

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

# Verify GPU availability
try:
    import torch
    gpu_count = torch.cuda.device_count()
    gpu_names = [torch.cuda.get_device_name(i) for i in range(gpu_count)]
    print(f"PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}")
    print(f"GPUs detected: {gpu_count}")
    for i, name in enumerate(gpu_names):
        vram_gb = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"  GPU {i}: {name}  ({vram_gb:.1f} GB VRAM)")
except ImportError:
    print("PyTorch not installed yet – will be installed in the next cell.")
    gpu_count = 0

import platform, psutil
ram_gb = psutil.virtual_memory().total / 1024**3
print(f"\nPython {platform.python_version()}  |  RAM {ram_gb:.0f} GB")

## 2 · Configuration

In [ ]:
# ── Project paths ──────────────────────────────────────────────────────────────
PROJECT_DIR  = Path("/kaggle/working/project")   # root of the cloned repo
OUTPUT_DIR   = Path("/kaggle/working/runs")       # where all run artefacts go

# ── Dataset ────────────────────────────────────────────────────────────────────
# Leave as None to let the script auto-detect from common Kaggle input paths,
# or set explicitly, e.g. Path("/kaggle/input/pcb-defect-dataset/data")
DATA_SOURCE  = None

# ── Training ───────────────────────────────────────────────────────────────────
# Supported: yolo11s  retinanet  faster_rcnn  cascade_rcnn  detr  deformable_detr
MODELS       = ["yolo11s", "retinanet", "faster_rcnn", "cascade_rcnn", "detr", "deformable_detr"]
EPOCHS       = 50
IMGSZ        = 640       # input image size (YOLO)
BATCH        = 32        # YOLO batch size per step (split across GPUs)
GPUS         = 2         # number of GPUs for MMDetection distributed training
YOLO_DEVICE  = "0,1"    # YOLO device string ("0" = GPU 0, "0,1" = both GPUs)
WORKERS      = 8         # dataloader workers

# ── Dependency installation ────────────────────────────────────────────────────
# Set to True on a fresh Kaggle session; False if deps are already installed
INSTALL_DEPS = True

print("Configuration:")
print(f"  PROJECT_DIR : {PROJECT_DIR}")
print(f"  OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"  DATA_SOURCE : {DATA_SOURCE}")
print(f"  MODELS      : {MODELS}")
print(f"  EPOCHS      : {EPOCHS}")
print(f"  GPUS        : {GPUS}")

## 3 · Install dependencies

In [ ]:
if INSTALL_DEPS:
    req_file = PROJECT_DIR / "scripts" / "requirements_train.txt"
    if req_file.exists():
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-U", "pip"],
            check=True
        )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)],
            check=True
        )
        print("Core dependencies installed.")
    else:
        print(f"requirements_train.txt not found at {req_file}")

    # mmcv requires openmim for GPU wheels
    mmdet_models = {"retinanet", "faster_rcnn", "cascade_rcnn", "detr", "deformable_detr"}
    if any(m in mmdet_models for m in MODELS):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "openmim"], check=True)
        subprocess.run([sys.executable, "-m", "mim", "install", "mmcv>=2.1.0,<2.2.0"], check=True)
        print("MMDetection ecosystem installed.")
else:
    print("Skipping dependency installation (INSTALL_DEPS=False).")

## 4 · Validate project structure

In [ ]:
assert PROJECT_DIR.exists(), f"Project directory not found: {PROJECT_DIR}"
assert (PROJECT_DIR / "scripts" / "train_kaggle.py").exists(), \
    f"train_kaggle.py not found under {PROJECT_DIR}/scripts/"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory ready: {OUTPUT_DIR}")

# Quick check for the dataset
data_candidates = [
    PROJECT_DIR / "data",
    Path("/kaggle/input/pcb-defect-dataset/data"),
    Path("/kaggle/input/pcb-defect-detection/data"),
    Path("/kaggle/input/pcb-defect/data"),
]
if DATA_SOURCE:
    data_candidates.insert(0, Path(DATA_SOURCE))

detected_data = None
for cand in data_candidates:
    if all((cand / d).exists() for d in ("train", "val", "test", "annotations_json")):
        detected_data = cand
        break

if detected_data:
    print(f"Dataset found at: {detected_data}")
else:
    print("WARNING: Could not auto-detect dataset. Set DATA_SOURCE in cell 2.")

## 5 · Dry-run (preview training plan)

In [ ]:
train_script = PROJECT_DIR / "scripts" / "train_kaggle.py"

dry_run_cmd = [
    sys.executable, str(train_script),
    "--models"] + MODELS + [
    "--epochs",      str(EPOCHS),
    "--imgsz",       str(IMGSZ),
    "--batch",       str(BATCH),
    "--gpus",        str(GPUS),
    "--yolo-device", YOLO_DEVICE,
    "--workers",     str(WORKERS),
    "--project",     str(OUTPUT_DIR),
    "--dry-run",
]
if DATA_SOURCE:
    dry_run_cmd += ["--data-source", str(DATA_SOURCE)]

print("[DRY RUN] Command:", " ".join(dry_run_cmd))
result = subprocess.run(dry_run_cmd, cwd=str(PROJECT_DIR), capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Dry run failed – fix errors above before starting training.")

## 6 · Train all models

Remove the `--models` entries you don't want to train, or run individual model cells below.

In [ ]:
train_cmd = [
    sys.executable, str(train_script),
    "--models"] + MODELS + [
    "--epochs",      str(EPOCHS),
    "--imgsz",       str(IMGSZ),
    "--batch",       str(BATCH),
    "--gpus",        str(GPUS),
    "--yolo-device", YOLO_DEVICE,
    "--workers",     str(WORKERS),
    "--project",     str(OUTPUT_DIR),
]
if DATA_SOURCE:
    train_cmd += ["--data-source", str(DATA_SOURCE)]

print("[TRAIN] Command:", " ".join(train_cmd))
subprocess.run(train_cmd, cwd=str(PROJECT_DIR), check=True)

## 7 · Train individual models (optional)

Use these cells if you want to train just one model at a time.

In [ ]:
# ── YOLO 11s only ──────────────────────────────────────────────────────────────
cmd = [
    sys.executable, str(train_script),
    "--models", "yolo11s",
    "--epochs",      str(EPOCHS),
    "--imgsz",       str(IMGSZ),
    "--batch",       str(BATCH),
    "--yolo-device", YOLO_DEVICE,
    "--workers",     str(WORKERS),
    "--project",     str(OUTPUT_DIR),
]
if DATA_SOURCE:
    cmd += ["--data-source", str(DATA_SOURCE)]
# subprocess.run(cmd, cwd=str(PROJECT_DIR), check=True)  # uncomment to run

In [ ]:
# ── RetinaNet only ────────────────────────────────────────────────────────────
cmd = [
    sys.executable, str(train_script),
    "--models", "retinanet",
    "--epochs",  str(EPOCHS),
    "--gpus",    str(GPUS),
    "--project", str(OUTPUT_DIR),
]
if DATA_SOURCE:
    cmd += ["--data-source", str(DATA_SOURCE)]
# subprocess.run(cmd, cwd=str(PROJECT_DIR), check=True)  # uncomment to run

In [ ]:
# ── Faster R-CNN only ─────────────────────────────────────────────────────────
cmd = [
    sys.executable, str(train_script),
    "--models", "faster_rcnn",
    "--epochs",  str(EPOCHS),
    "--gpus",    str(GPUS),
    "--project", str(OUTPUT_DIR),
]
if DATA_SOURCE:
    cmd += ["--data-source", str(DATA_SOURCE)]
# subprocess.run(cmd, cwd=str(PROJECT_DIR), check=True)  # uncomment to run

In [ ]:
# ── Cascade R-CNN only ────────────────────────────────────────────────────────
cmd = [
    sys.executable, str(train_script),
    "--models", "cascade_rcnn",
    "--epochs",  str(EPOCHS),
    "--gpus",    str(GPUS),
    "--project", str(OUTPUT_DIR),
]
if DATA_SOURCE:
    cmd += ["--data-source", str(DATA_SOURCE)]
# subprocess.run(cmd, cwd=str(PROJECT_DIR), check=True)  # uncomment to run

In [ ]:
# ── DETR only ─────────────────────────────────────────────────────────────────
cmd = [
    sys.executable, str(train_script),
    "--models", "detr",
    "--epochs",  str(EPOCHS),
    "--gpus",    str(GPUS),
    "--project", str(OUTPUT_DIR),
]
if DATA_SOURCE:
    cmd += ["--data-source", str(DATA_SOURCE)]
# subprocess.run(cmd, cwd=str(PROJECT_DIR), check=True)  # uncomment to run

In [ ]:
# ── Deformable DETR only ──────────────────────────────────────────────────────
cmd = [
    sys.executable, str(train_script),
    "--models", "deformable_detr",
    "--epochs",  str(EPOCHS),
    "--gpus",    str(GPUS),
    "--project", str(OUTPUT_DIR),
]
if DATA_SOURCE:
    cmd += ["--data-source", str(DATA_SOURCE)]
# subprocess.run(cmd, cwd=str(PROJECT_DIR), check=True)  # uncomment to run

## 8 · Inspect results

In [ ]:
import json

print(f"Run artefacts in {OUTPUT_DIR}:")
for p in sorted(OUTPUT_DIR.rglob("*.json")):
    print(" ", p.relative_to(OUTPUT_DIR))

# Show YOLO results summary if available
yolo_results = OUTPUT_DIR / "yolo11s" / "results.csv"
if yolo_results.exists():
    import csv
    rows = list(csv.DictReader(yolo_results.open()))
    if rows:
        last = rows[-1]
        print("\nYOLO final epoch metrics:")
        for k, v in last.items():
            print(f"  {k.strip():35s} {v.strip()}")